In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, hamming_loss, f1_score
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# For deep learning model
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizer, DistilBertModel
from transformers import logging
logging.set_verbosity_error()

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

print("Loading and preprocessing data...")
# 1. Load and preprocess data
df = pd.read_csv('clean_movie_summary_genre.csv')

# Convert genres to list
df['genres'] = df['genres'].str.split(',').apply(lambda x: [g.strip() for g in x])

# Display data stats
print(f"Dataset shape: {df.shape}")
print(f"Number of unique genres: {len(set([g for sublist in df['genres'] for g in sublist]))}")

# 2. Filter out rare genres and keep most common ones
all_genres = [genre for sublist in df['genres'] for genre in sublist]
genre_counts = Counter(all_genres)
print("\nTop 15 genres by frequency:")
for genre, count in genre_counts.most_common(15):
    print(f"{genre}: {count}")

# Keep genres that appear at least 20 times (adjust based on your dataset)
min_genre_count = 20
valid_genres = [genre for genre, count in genre_counts.items() if count >= min_genre_count]
print(f"\nKeeping {len(valid_genres)} genres that appear at least {min_genre_count} times")

# Filter the dataset
df['genres'] = df['genres'].apply(lambda x: [g for g in x if g in valid_genres])
df = df[df['genres'].apply(len) > 0]  # Remove rows with no valid genres
print(f"Dataset shape after filtering: {df.shape}")

# 3. Multi-label binarization
mlb = MultiLabelBinarizer(classes=valid_genres)
y = mlb.fit_transform(df['genres'])

# 4. Train-test-validation split (with stratified sampling approximation)
# First split into train and temp
X_train, X_temp, y_train, y_temp = train_test_split(
    df['clean_summary'].values, 
    y,
    test_size=0.3, 
    random_state=42
)

# Then split temp into validation and test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42
)

print(f"Train set: {len(X_train)} samples")
print(f"Validation set: {len(X_val)} samples")
print(f"Test set: {len(X_test)} samples")

# 5. Define a custom dataset for PyTorch
class MovieDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.float)
        }

# 6. Define the model architecture
class GenreClassifier(nn.Module):
    def __init__(self, n_classes):
        super(GenreClassifier, self).__init__()
        self.distilbert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.classifier = nn.Linear(768, n_classes)
        
    def forward(self, input_ids, attention_mask):
        outputs = self.distilbert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        
        # Use the CLS token representation
        pooled_output = outputs.last_hidden_state[:, 0]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits

# 7. Training function
def train_model(model, train_dataloader, val_dataloader, device, epochs=3):
    optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
    criterion = nn.BCEWithLogitsLoss()
    
    best_val_f1 = 0
    history = {'train_loss': [], 'val_loss': [], 'val_f1': []}
    
    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")
        
        # Training phase
        model.train()
        train_loss = 0
        for batch in train_dataloader:
            optimizer.zero_grad()
            
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs, labels)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            
        avg_train_loss = train_loss / len(train_dataloader)
        history['train_loss'].append(avg_train_loss)
        print(f"Training loss: {avg_train_loss:.4f}")
        
        # Validation phase
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []
        
        with torch.no_grad():
            for batch in val_dataloader:
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)
                
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs, labels)
                
                val_loss += loss.item()
                
                # Convert logits to predictions (using 0.5 threshold)
                preds = torch.sigmoid(outputs) > 0.5
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())
                
        avg_val_loss = val_loss / len(val_dataloader)
        val_f1 = f1_score(all_labels, all_preds, average='micro')
        
        history['val_loss'].append(avg_val_loss)
        history['val_f1'].append(val_f1)
        
        print(f"Validation loss: {avg_val_loss:.4f}")
        print(f"Validation F1 score: {val_f1:.4f}")
        
        # Save best model
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), 'best_genre_model.pt')
            print("Saved best model!")
    
    return history

# 8. Evaluation function
def evaluate_model(model, test_dataloader, device, threshold=0.5):
    model.eval()
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs)
            preds = probs > threshold
            
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    
    # Convert to numpy arrays
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    hamming = hamming_loss(all_labels, all_preds)
    micro_f1 = f1_score(all_labels, all_preds, average='micro')
    macro_f1 = f1_score(all_labels, all_preds, average='macro')
    
    print(f"Accuracy: {accuracy:.4f}")
    print(f"Hamming Loss: {hamming:.4f}")
    print(f"Micro F1 Score: {micro_f1:.4f}")
    print(f"Macro F1 Score: {macro_f1:.4f}")
    
    # Classification report for each genre
    report = classification_report(
        all_labels, 
        all_preds, 
        target_names=mlb.classes_,
        zero_division=0
    )
    print("\nClassification Report:")
    print(report)
    
    return {
        'accuracy': accuracy,
        'hamming_loss': hamming,
        'micro_f1': micro_f1,
        'macro_f1': macro_f1,
        'predictions': all_preds,
        'labels': all_labels
    }

# 9. Prediction function
class GenrePredictor:
    def __init__(self, model_path, tokenizer, mlb, device, threshold=0.5):
        self.model = GenreClassifier(len(mlb.classes_))
        self.model.load_state_dict(torch.load(model_path, map_location=device))
        self.model.to(device)
        self.model.eval()
        
        self.tokenizer = tokenizer
        self.mlb = mlb
        self.device = device
        self.threshold = threshold
        
    def predict(self, text):
        # Tokenize
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=512,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Move to device
        input_ids = encoding['input_ids'].to(self.device)
        attention_mask = encoding['attention_mask'].to(self.device)
        
        # Predict
        with torch.no_grad():
            outputs = self.model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.sigmoid(outputs).cpu().numpy()[0]
        
        # Get predicted genres with probabilities
        genre_probs = {
            self.mlb.classes_[i]: float(probs[i]) 
            for i in range(len(self.mlb.classes_))
        }
        
        # Sort by probability
        sorted_genres = sorted(
            genre_probs.items(), 
            key=lambda x: x[1], 
            reverse=True
        )
        
        # Filter by threshold
        predicted_genres = [
            genre for genre, prob in sorted_genres 
            if prob > self.threshold
        ]
        
        return {
            'predicted_genres': predicted_genres,
            'genre_probabilities': sorted_genres
        }

# 10. Visualization functions
def plot_genre_distribution(genres_list):
    """Plot distribution of top genres in the dataset"""
    plt.figure(figsize=(12, 6))
    genre_counts = Counter([g for sublist in genres_list for g in sublist])
    top_genres = dict(genre_counts.most_common(15))
    
    plt.bar(top_genres.keys(), top_genres.values())
    plt.xticks(rotation=45, ha='right')
    plt.title('Top 15 Movie Genres')
    plt.tight_layout()
    plt.savefig('genre_distribution.png')
    plt.close()

def plot_training_history(history):
    """Plot training and validation metrics"""
    plt.figure(figsize=(12, 5))
    
    plt.subplot(1, 2, 1)
    plt.plot(history['train_loss'], label='Train Loss')
    plt.plot(history['val_loss'], label='Validation Loss')
    plt.title('Training and Validation Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    
    plt.subplot(1, 2, 2)
    plt.plot(history['val_f1'], label='Validation F1')
    plt.title('Validation F1 Score')
    plt.xlabel('Epoch')
    plt.ylabel('F1 Score')
    plt.legend()
    
    plt.tight_layout()
    plt.savefig('training_history.png')
    plt.close()

def plot_confusion_matrix(y_true, y_pred, class_names, n_classes=10):
    """Plot confusion matrix for top n_classes"""
    # Calculate class-wise metrics
    class_metrics = []
    for i, class_name in enumerate(class_names):
        tp = np.sum((y_true[:, i] == 1) & (y_pred[:, i] == 1))
        fp = np.sum((y_true[:, i] == 0) & (y_pred[:, i] == 1))
        fn = np.sum((y_true[:, i] == 1) & (y_pred[:, i] == 0))
        tn = np.sum((y_true[:, i] == 0) & (y_pred[:, i] == 0))
        
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0
        f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
        
        class_metrics.append({
            'class': class_name,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'support': np.sum(y_true[:, i])
        })
    
    # Sort by support (number of examples) and select top n
    class_metrics.sort(key=lambda x: x['support'], reverse=True)
    top_classes = class_metrics[:n_classes]
    
    # Create a dataframe for visualization
    metrics_df = pd.DataFrame(top_classes)
    
    plt.figure(figsize=(10, 6))
    sns.heatmap(
        metrics_df[['precision', 'recall', 'f1']].values,
        annot=True,
        cmap='Blues',
        fmt='.3f',
        xticklabels=['Precision', 'Recall', 'F1'],
        yticklabels=metrics_df['class']
    )
    plt.title(f'Performance Metrics for Top {n_classes} Movie Genres')
    plt.tight_layout()
    plt.savefig('genre_metrics.png')
    plt.close()

# Main execution
if __name__ == "__main__":
    # Check for GPU
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}")
    
    # Plot genre distribution
    plot_genre_distribution(df['genres'])
    
    # Initialize tokenizer
    print("\nLoading DistilBERT tokenizer...")
    tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
    
    # Create datasets
    print("Creating datasets...")
    batch_size = 16
    
    train_dataset = MovieDataset(X_train, y_train, tokenizer)
    val_dataset = MovieDataset(X_val, y_val, tokenizer)
    test_dataset = MovieDataset(X_test, y_test, tokenizer)
    
    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size)
    test_dataloader = DataLoader(test_dataset, batch_size=batch_size)
    
    # Initialize model
    print("Initializing model...")
    model = GenreClassifier(len(mlb.classes_))
    model.to(device)
    
    # Train model
    print("\nStarting training...")
    history = train_model(model, train_dataloader, val_dataloader, device, epochs=3)
    
    # Plot training history
    plot_training_history(history)
    
    # Load best model for evaluation
    print("\nLoading best model for evaluation...")
    best_model = GenreClassifier(len(mlb.classes_))
    best_model.load_state_dict(torch.load('best_genre_model.pt', map_location=device))
    best_model.to(device)
    
    # Evaluate on test set
    print("\nEvaluating on test set:")
    eval_results = evaluate_model(best_model, test_dataloader, device)
    
    # Plot confusion matrix for top genres
    plot_confusion_matrix(eval_results['labels'], eval_results['predictions'], mlb.classes_)
    
    # Save model artifacts
    print("\nSaving model artifacts...")
    torch.save(best_model.state_dict(), 'genre_classifier_distilbert.pt')
    joblib.dump(mlb, 'genre_classifier_mlb.pkl')
    
    # Create predictor
    predictor = GenrePredictor(
        model_path='best_genre_model.pt',
        tokenizer=tokenizer,
        mlb=mlb,
        device=device,
        threshold=0.3  # Can adjust this threshold
    )
    
    # Example predictions
    print("\nExample predictions:")
    test_summaries = [
        "A detective solves mysterious crimes in a futuristic city populated by robots.",
        "A group of astronauts travel to Mars and encounter alien life forms that threaten their mission.",
        "A young couple falls in love despite their families' opposition based on ancient traditions.",
        "Superheroes with extraordinary powers join forces to save the world from a powerful villain.",
        "A talented chef struggles to keep his restaurant afloat while dealing with personal issues."
    ]
    
    for summary in test_summaries:
        print(f"\nSummary: {summary}")
        result = predictor.predict(summary)
        print(f"Predicted genres: {result['predicted_genres']}")
        print("Top genre probabilities:")
        for genre, prob in result['genre_probabilities'][:3]:
            print(f"  - {genre}: {prob:.4f}")

Loading and preprocessing data...
Dataset shape: (41796, 3)
Number of unique genres: 363

Top 15 genres by frequency:
Drama: 19135
Comedy: 10468
Romance Film: 6666
Thriller: 6530
Action: 5869
World cinema: 5153
Crime Fiction: 4277
Horror: 4083
Black-and-white: 3731
Indie: 3668
Action/Adventure: 3553
Adventure: 3248
Family Film: 3219
Short Film: 3192
Romantic drama: 2572

Keeping 216 genres that appear at least 20 times
Dataset shape after filtering: (41777, 3)
Train set: 29243 samples
Validation set: 6267 samples
Test set: 6267 samples
Using device: cpu

Loading DistilBERT tokenizer...
Creating datasets...
Initializing model...

Starting training...

Epoch 1/3
Training loss: 0.0991
Validation loss: 0.0535
Validation F1 score: 0.2175
Saved best model!

Epoch 2/3
Training loss: 0.0501
Validation loss: 0.0465
Validation F1 score: 0.3757
Saved best model!

Epoch 3/3
Training loss: 0.0439
Validation loss: 0.0439
Validation F1 score: 0.4148
Saved best model!

Loading best model for evaluatio